# Recursive Wishart on Colab + Google Drive\n\nThe pipeline computes on fast `/content` scratch and checkpoints durable artifacts to Google Drive. Edit the paths in the configuration cell before running.

In [ ]:
from pathlib import Path\nimport sys\n\nIN_COLAB = 'google.colab' in sys.modules\nprint({'in_colab': IN_COLAB, 'cwd': str(Path.cwd())})\n

In [ ]:
# Install from the checked-out repository.\n%pip install -q -c requirements/constraints.txt -e '.[wishart,notebook]'\n

In [ ]:
# Mount Drive explicitly in the notebook frontend.\nif IN_COLAB:\n    from google.colab import drive\n    drive.mount('/content/drive')\nelse:\n    print('Not in Colab; Drive mount skipped.')\n

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/SemanticMap/semgraphex')\nCONFIG = Path('configs/wishart_conceptnet_colab.yaml')\n\n# Choose exactly one source. Relative paths are resolved under DRIVE_ROOT.\nPREPARED_DRIVE = Path('prepared/prepare-REPLACE_ME')\nDATASET_DRIVE = None  # e.g. Path('data/conceptnet_en_100k.tsv')\n\nRUN_NAME = 'wishart-typed-wl-colab-01'\nKEEP_SCRATCH = True\nRUN_PIPELINE = False  # Change to True after checking paths.\n\nprint({'drive_root': str(DRIVE_ROOT), 'prepared': str(PREPARED_DRIVE), 'run_name': RUN_NAME})\n

In [ ]:
import subprocess\n\ncommand = [\n    'semmap-wishart-colab',\n    '--config', str(CONFIG),\n    '--drive-root', str(DRIVE_ROOT),\n    '--run-name', RUN_NAME,\n]\nif PREPARED_DRIVE is not None:\n    command += ['--prepared-drive', str(PREPARED_DRIVE)]\nelif DATASET_DRIVE is not None:\n    command += ['--dataset-drive', str(DATASET_DRIVE)]\nelse:\n    raise ValueError('Set PREPARED_DRIVE or DATASET_DRIVE')\nif KEEP_SCRATCH:\n    command.append('--keep-scratch')\n\nprint(' '.join(command))\nif RUN_PIPELINE:\n    subprocess.run(command, check=True)\nelse:\n    print('Dry setup only. Set RUN_PIPELINE=True to start.')\n

In [ ]:
# Inspect the durable Drive checkpoint/result after a run.\ndrive_run = DRIVE_ROOT / 'runs' / RUN_NAME\nif drive_run.exists():\n    print(sorted(p.name for p in drive_run.iterdir()))\n    checkpoint = drive_run / 'DRIVE_CHECKPOINT.json'\n    if checkpoint.exists():\n        print(checkpoint.read_text()[:4000])\nelse:\n    print('Drive run directory does not exist yet.')\n